# Batch Correction Visualization Pipeline

This notebook provides a step-by-step interactive visualization for evaluating **Batch Correction** results.
It compares **Time-Corrected + Arcsinh** data (Before Batch Correction) with **Batch-Corrected** data (After Batch Correction).

#### This is for the visualization of the Batch-corrected results

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os
import sys
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from collections import defaultdict
import collections.abc
collections.MutableMapping = collections.abc.MutableMapping
import scanpy as sc
import seaborn as sns
import anndata as ad
from matplotlib.lines import Line2D
%matplotlib inline
# Setup path to import local CyTof_BatchCorrection package
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils import *
from src.batch_correction_visualization import *
from src.time_correction_visualization import *
print(f"Project root: {project_root}")

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────────
study_config_path = "../results/cohort_1/study_config_cohort1_D24_Panel2.json"

# Set BOTH to process one panel/dataset at a time (recommended — keeps memory low).
# Leave None to auto-select: dataset → study config value, panel → first found.
target_dataset_override = None     # e.g. "D0"
target_panel_override   = "Panel2" #"Panel1" # e.g. "Panel1" or "Panel2"
# ─────────────────────────────────────────────────────────────────────────────

with open(study_config_path) as f:
    study_config = json.load(f)

config = get_config(config_dict=study_config["cytof_config"])
ctx    = CyTOFContext(config)

with open(study_config["outputs"]["markers_path"]) as f:
    markers = json.load(f)

target_dataset = target_dataset_override or study_config["dataset"]
anchor_name    = ctx.anchors.get(target_dataset)

raw_path   = study_config["outputs"]["matrix_path"]
final_path = study_config["outputs"]["matrix_corrected_path"]

df_raw = pd.read_csv(raw_path)
df_a   = pd.read_csv(final_path)

# Filter to target dataset
if target_dataset and "dataset" in df_raw.columns:
    df_raw = df_raw[df_raw["dataset"] == target_dataset].copy()
    df_a   = df_a[df_a["dataset"] == target_dataset].copy()

# Select the panel to analyse — default to first available
available_panels = sorted(df_raw["panel"].unique()) if "panel" in df_raw.columns else [study_config.get("panel", "Panel1")]
target_panel = target_panel_override or available_panels[0]

# Keep ONLY the target panel in memory
df_raw = df_raw[df_raw["panel"] == target_panel].copy()
df_a   = df_a[df_a["panel"] == target_panel].copy()

# Apply arcsinh to raw data
numeric_markers = [m for m in markers if m in df_raw.columns]
df_b = arcsinh(df_raw, numeric_markers)
del df_raw

print(f"Dataset  : {target_dataset}")
print(f"Panel    : {target_panel}")
print(f"Anchor   : {anchor_name}")
print(f"Before   : {len(df_b)} cells | After: {len(df_a)} cells")
print(f"Markers ({len(markers)}): {markers[:5]}...")

In [ ]:
# --- GENERATE & LOAD STUDY-SPECIFIC COLORS ---
_results_dir     = os.path.dirname(study_config["outputs"]["matrix_corrected_path"])
_study_name      = study_config.get("study_name", "study")
colors_save_path = os.path.join(_results_dir, f"colors_dict_{_study_name}.pkl")

if df_a is not None and not os.path.exists(colors_save_path):
    print("\n===== GENERATING GLOBAL COLORS FROM GATED DATA =====")
    all_cell_types = set()
    tc_dir = os.path.join(ctx.output_dir, "time_corrected")
    if os.path.exists(tc_dir):
        for f in glob(os.path.join(tc_dir, "*.csv")):
            try:
                df_temp = pd.read_csv(f, usecols=["cell_type"])
                all_cell_types.update(df_temp["cell_type"].dropna().unique())
            except: pass
    cols = ["celltypes", "panel", "plate", "sample_name", "dataset"]
    temp_df = df_a[cols].drop_duplicates().copy()
    for col in cols:
        if col not in temp_df.columns:
            temp_df[col] = np.nan
    temp_df["cell_type"] = pd.Series(list(all_cell_types))
    generate_and_save_colors(df=temp_df, save_path=colors_save_path)
elif os.path.exists(colors_save_path):
    print(f"[INFO] Using existing colors from {colors_save_path}")

global_colors = {}
if os.path.exists(colors_save_path):
    with open(colors_save_path, "rb") as f_in:
        global_colors = pickle.load(f_in)
    print(f"[INFO] Global colors loaded ({len(global_colors)} keys)")

In [ ]:
import gc

factors = [c for c in ["plate", "group", "celltypes", "Time"] if c in df_b.columns]

# df_b and df_a are already filtered to target_panel — no loop needed
if df_b.empty or df_a.empty:
    print(f"[SKIP] No data found for panel {target_panel}")
else:
    # Restrict to true biological markers present in BOTH before and after DataFrames.
    panel_markers = [
        m for m in markers
        if m in df_b.columns
        and m in df_a.columns
        and m in ctx.true_markers
        and df_b[m].notna().any()
    ]
    print(f"Panel    : {target_panel}  |  Dataset: {target_dataset}")
    print(f"Markers  : {len(panel_markers)} | Before : {len(df_b)} | After : {len(df_a)}")

    # Per-panel colors
    current_colors = {}
    if global_colors and os.path.exists(colors_save_path):
        current_colors = load_and_subset_colors(df_a, colors_save_path)

    # Output directories
    base_panel_dir  = os.path.join(ctx.output_dir, target_dataset, target_panel)
    path_Variance   = os.path.join(base_panel_dir, "Batch_Variance")
    path_EMD        = os.path.join(base_panel_dir, "Batch_EMD")
    path_MarkerCorr = os.path.join(base_panel_dir, "Batch_MarkerCorrelation")
    for _d in [path_Variance, path_EMD, path_MarkerCorr]:
        os.makedirs(_d, exist_ok=True)

    # ── 1. PCA Variance Analysis ──────────────────────────────────────────
    print(f"\n[ANALYZE] Factor PCA Variance...")
    var_b = variation_pca(df_b, markers=panel_markers, group_col="condition",
                          info_cols=factors, panel=target_panel)
    var_a = variation_pca(df_a, markers=panel_markers, group_col="condition",
                          info_cols=factors, panel=target_panel)
    print("Before correction:"); print(var_b)
    print("After  correction:"); print(var_a)

    # ── 2. Anchor Signal Alignment (across plates, before correction) ─────
    print(f"\n[ANALYZE] Anchor Signal Alignment across plates...")
    visualize_anchor_across_plates(df=df_b, panel=target_panel,
                                   markers=panel_markers, save_dir=None,
                                   anchor_name=anchor_name)
    plt.show()

    # ── 3. Anchor Before / After comparison ───────────────────────────────
    df_anc_b = df_b[df_b["sample_name"] == anchor_name].copy()
    df_anc_a = df_a[df_a["sample_name"] == anchor_name].copy()
    if not df_anc_b.empty and not df_anc_a.empty:
        print(f"\n[ANALYZE] Anchor before/after distribution comparison...")
        plot_signal_distributions(df_before=df_anc_b, df_after=df_anc_a,
                                  panel=target_panel, markers=panel_markers,
                                  save_dir=None)
        plt.show()
    else:
        print(f"  [SKIP] Anchor '{anchor_name}' not found for {target_panel}")
    del df_anc_b, df_anc_a

    # ── 4. UMAPs ──────────────────────────────────────────────────────────
    print(f"\n[ANALYZE] UMAP — Plate Effect...")
    umap_plate_effect(df_b, df_a, markers=panel_markers,
                      subsample=50000, save_path=None)
    plt.show()

    print(f"\n[ANALYZE] UMAP — Batch Evaluation...")
    umap_batch_evaluation(df_raw=df_b, df_corrected=df_a,
                          markers=panel_markers, save_path=None)
    plt.show()

    print(f"\n[ANALYZE] UMAP — Celltype by Plate...")
    umap_celltype_by_plate(df_raw=df_b, df_corrected=df_a,
                           markers=panel_markers, subsample=50000, save_path=None)
    plt.show()

    # ── 5. EMD / MAD ──────────────────────────────────────────────────────
    print(f"\n[ANALYZE] Computing EMD/MAD Scores...")
    cluster_col = "celltypes" if "celltypes" in df_b.columns else None
    emd_df, _, mad_score = compute_emd_mad(
        df_raw=df_b, df_corr=df_a,
        markers=panel_markers,
        cluster_col=cluster_col,
        anchor=anchor_name,
        n_sample=50000,
        save_dir=path_EMD,
    )
    if emd_df is not None:
        avg_mad = mad_score["MAD_score"].median() if isinstance(mad_score, pd.DataFrame) else mad_score
        print(f"  MAD Score (median): {avg_mad:.4f}")
        plot_emd(emd_df, mad_score, save_path=path_EMD)
        plot_emd_per_marker(emd_df, save_path=path_EMD)
        plt.show()

    # ── 6. Marker-level Variance Boxplot ──────────────────────────────────
    print(f"\n[ANALYZE] Marker-level variance boxplots...")
    mvar_b = variation_marker(df_b, markers=panel_markers, group_col=None,
                              info_cols=factors, panel=target_panel)
    mvar_a = variation_marker(df_a, markers=panel_markers, group_col=None,
                              info_cols=factors, panel=target_panel)
    mvar_b.to_csv(os.path.join(path_Variance,
        f"variance_marker_raw_before_{target_dataset}_{target_panel}_Batch.csv"), index=False)
    mvar_a.to_csv(os.path.join(path_Variance,
        f"variance_marker_raw_after_{target_dataset}_{target_panel}_Batch.csv"), index=False)
    plot_variance_marker_boxplot(
        var_marker_raw=mvar_b, var_marker_corr=mvar_a,
        save_path=os.path.join(path_Variance,
            f"variance_marker_boxplot_{target_dataset}_{target_panel}_Batch.png"),
        use_log=True
    )
    plt.show()

    # ── 7. Batch Correction Metrics ────────────────────────────────────────
    print(f"\n[ANALYZE] Batch Correction Metrics...")
    try:
        metrics_before, metrics_after = compute_metrics(
            df_after=df_a, df_before=df_b,
            markers=panel_markers, batch_key="plate",
            label_key="celltypes", n_iterations=20,
        )
        plot_metrics_comparison(metrics_before, metrics_after,
                                panel=target_panel, save_dir=path_Variance)
        plt.show()
    except Exception as _e:
        print(f"  [WARN] Metrics skipped: {_e}")

    # ── 8. Marker-Marker Correlation ───────────────────────────────────────
    print(f"\n[ANALYZE] Marker-Marker Correlation...")
    marker_corr(df_before=df_b, df_after=df_a,
                markers=panel_markers, save_dir=path_MarkerCorr,
                group_col="group")
    plt.show()

    gc.collect()
    print(f"\n[DONE] Finished panel: {target_panel}")

In [ ]:
import gc

# Memory cleanup before cross-panel analysis.
# df_b and df_a (all panels) are freed here; the cross-panel cell below re-reads
# the matrices from the study config paths.
try:
    del df_b, df_a
except NameError:
    pass
gc.collect()
print("[MEMORY] Panel data cleared. Ready for Cross-Panel / Cross-Dataset Analysis.")

In [ ]:
import gc

# ── Cross-Panel Variance Analysis ─────────────────────────────────────────────
# Re-reads BOTH panels from the study config paths.
# (df_b / df_a above hold only one panel to keep memory usage low.)

# ── USER INPUT ────────────────────────────────────────────────────────────────
p1_name_override = None   # e.g. "Panel1"
p2_name_override = None   # e.g. "Panel2"
# ─────────────────────────────────────────────────────────────────────────────

try:
    del df_b, df_a
except NameError:
    pass
gc.collect()

_before_path = study_config["outputs"]["matrix_path"]
_after_path  = study_config["outputs"]["matrix_corrected_path"]

# Save cross-panel figures under Correction/{dataset}/cross_panel_visualization/
# so D0 and D24 each get their own folder (consistent with per-panel Batch_* folders).
vis_save_dir = os.path.join(ctx.output_dir, target_dataset, "cross_panel_visualization")
os.makedirs(vis_save_dir, exist_ok=True)
print(f"[INFO] Cross-panel figures → {vis_save_dir}")

print("[INFO] Re-reading both panels for cross-panel analysis...")
_df_raw  = pd.read_csv(_before_path)
df_all_a = pd.read_csv(_after_path)

if target_dataset and "dataset" in _df_raw.columns:
    _df_raw  = _df_raw[_df_raw["dataset"] == target_dataset].copy()
    df_all_a = df_all_a[df_all_a["dataset"] == target_dataset].copy()

# Recompute numeric_markers from the full (both-panel) raw data
_all_numeric = [m for m in markers if m in _df_raw.columns]
df_all_b = arcsinh(_df_raw, _all_numeric)
del _df_raw

available_panels = sorted(df_all_b["panel"].unique()) if "panel" in df_all_b.columns else []
print(f"[INFO] Available panels: {available_panels}")

if len(available_panels) < 2:
    print("[SKIP] Cross-panel analysis requires at least 2 panels. Skipping.")
else:
    p1_name = p1_name_override or available_panels[0]
    p2_name = p2_name_override or available_panels[1]

    factors = [c for c in ["plate", "group", "celltypes", "Time"] if c in df_all_b.columns]

    # Resolve pheno_markers for UMAPs (same logic as pipeline)
    _pheno = ctx.pheno_markers or study_config.get("cytof_config", {}).get("pheno_markers")

    try:
        # --- Panel 1 ---
        print(f"\n[ANALYZE] PCA for {p1_name}...")
        df_p1_b = df_all_b[df_all_b["panel"] == p1_name].copy()
        df_p1_a = df_all_a[df_all_a["panel"] == p1_name].copy()

        if df_p1_b.empty or df_p1_a.empty:
            raise ValueError(f"No data for {p1_name} in one or both matrices.")

        markers_p1 = [c for c in df_p1_b.columns if c not in ctx.meta_cols and c != "celltypes"]
        var_p1_b = variation_pca(df_p1_b, markers=markers_p1, group_col="condition", info_cols=factors, panel=p1_name)
        var_p1_a = variation_pca(df_p1_a, markers=markers_p1, group_col="condition", info_cols=factors, panel=p1_name)
        c_var_p1_b, c_var_p1_a = compute_celltype_variance(df_p1_b, df_p1_a, batch_col="plate", ctx=ctx)

        # --- Panel 2 ---
        print(f"\n[ANALYZE] PCA for {p2_name}...")
        df_p2_b = df_all_b[df_all_b["panel"] == p2_name].copy()
        df_p2_a = df_all_a[df_all_a["panel"] == p2_name].copy()

        if df_p2_b.empty or df_p2_a.empty:
            raise ValueError(f"No data for {p2_name} in one or both matrices.")

        markers_p2 = [c for c in df_p2_b.columns if c not in ctx.meta_cols and c != "celltypes"]
        var_p2_b = variation_pca(df_p2_b, markers=markers_p2, group_col="condition", info_cols=factors, panel=p2_name)
        var_p2_a = variation_pca(df_p2_a, markers=markers_p2, group_col="condition", info_cols=factors, panel=p2_name)
        c_var_p2_b, c_var_p2_a = compute_celltype_variance(df_p2_b, df_p2_a, batch_col="plate", ctx=ctx)

        # Pheno-filtered marker lists for UMAP (intersect with shared markers)
        if _pheno:
            umap_m1 = [m for m in _pheno if m in df_p1_b.columns]
            umap_m2 = [m for m in _pheno if m in df_p2_b.columns]
        else:
            umap_m1 = markers_p1
            umap_m2 = markers_p2

        # --- Cross-Panel Plots (saved to vis_save_dir) ---
        print(f"\n[POST-PROCESS] Cross-panel variance: {p1_name} vs {p2_name}")
        plot_variance_bar(var_p1_b, var_p1_a, var_p2_b, var_p2_a,
                          save_path=os.path.join(vis_save_dir, "global_variance_bar.png"))
        plt.show()

        plot_variance_bar_celltype(c_var_p1_b, c_var_p1_a, c_var_p2_b, c_var_p2_a,
                                   save_path=os.path.join(vis_save_dir, "celltype_variance_bar.png"))
        plt.show()

        print(f"\n[POST-PROCESS] Shared marker correction visualization...")
        visualize_shared_marker_correction(
            df_p1_b, df_p1_a, df_p2_b, df_p2_a,
            umap_m1, umap_m2,
            save_dir=vis_save_dir,
        )

        marker_expression_correlation_plot(
            df_p1_b, df_p1_a, df_p2_b, df_p2_a,
            save_dir=vis_save_dir
        )

        del df_p1_b, df_p1_a, df_p2_b, df_p2_a
        gc.collect()
        print(f"\n[DONE] Cross-panel figures saved to: {vis_save_dir}")

    except Exception as e:
        import traceback
        print(f"[ERROR] Cross-panel visualization failed: {e}")
        traceback.print_exc()

del df_all_b, df_all_a
gc.collect()

## Step 8 — Cross-Dataset Comparison

Compare batch-corrected outputs from **two different study configs** (e.g. D0 vs D24, or cohort_1 vs cohort_2).  
Only markers shared between both datasets are used. Set `panel_filter` to restrict to a specific panel, or leave `None` to use all panels.

In [ ]:
import gc

# Step 8: Cross-Dataset Comparison
# Loads corrected matrices from two study configs and compares shared markers.

# ── USER INPUT ────────────────────────────────────────────────────────────────
study_config_path_ds1 = "../results/cohort_1/study_config_cohort1_D0.json"
study_config_path_ds2 = "../results/cohort_1/study_config_cohort1_D24.json"

# Optional: restrict to a single panel (e.g. "Panel1"). None = use all panels.
panel_filter = None

# Number of cells to subsample per dataset for heavy plots (None = no limit)
cross_ds_subsample = 50000
# ─────────────────────────────────────────────────────────────────────────────

gc.collect()

with open(study_config_path_ds1) as _f:
    _sc1 = json.load(_f)
with open(study_config_path_ds2) as _f:
    _sc2 = json.load(_f)

_ds1_label = _sc1.get("study_name") or _sc1.get("dataset", "Dataset1")
_ds2_label = _sc2.get("study_name") or _sc2.get("dataset", "Dataset2")

print(f"[INFO] Loading corrected matrices...")
print(f"  DS1: {_ds1_label}  →  {_sc1['outputs']['matrix_corrected_path']}")
print(f"  DS2: {_ds2_label}  →  {_sc2['outputs']['matrix_corrected_path']}")

df_ds1 = pd.read_csv(_sc1["outputs"]["matrix_corrected_path"])
df_ds2 = pd.read_csv(_sc2["outputs"]["matrix_corrected_path"])

# Load per-study markers (biological markers only)
with open(_sc1["outputs"]["markers_path"]) as _f:
    _markers1 = json.load(_f)
with open(_sc2["outputs"]["markers_path"]) as _f:
    _markers2 = json.load(_f)

# Optionally filter to a single panel
if panel_filter:
    if "panel" in df_ds1.columns:
        df_ds1 = df_ds1[df_ds1["panel"] == panel_filter].copy()
    if "panel" in df_ds2.columns:
        df_ds2 = df_ds2[df_ds2["panel"] == panel_filter].copy()
    print(f"[INFO] Panel filter applied: {panel_filter}")

# Shared markers present in both datasets
_shared_markers = sorted(
    set(_markers1) & set(_markers2) & set(df_ds1.columns) & set(df_ds2.columns)
)
print(f"[INFO] Shared markers ({len(_shared_markers)}): {_shared_markers[:8]}...")

if not _shared_markers:
    print("[SKIP] No shared markers found between the two datasets.")
else:
    # Subsample for performance
    if cross_ds_subsample and len(df_ds1) > cross_ds_subsample:
        df_ds1 = df_ds1.sample(cross_ds_subsample, random_state=42)
    if cross_ds_subsample and len(df_ds2) > cross_ds_subsample:
        df_ds2 = df_ds2.sample(cross_ds_subsample, random_state=42)

    print(f"[INFO] Cells: {_ds1_label}={len(df_ds1)}, {_ds2_label}={len(df_ds2)}")

    # ── 1. Shared-marker distribution comparison (violin / ridge) ──────────────
    n_cols = min(4, len(_shared_markers))
    n_rows = int(np.ceil(len(_shared_markers) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
    axes = np.array(axes).flatten()

    for i, marker in enumerate(_shared_markers):
        ax = axes[i]
        vals1 = df_ds1[marker].dropna().values
        vals2 = df_ds2[marker].dropna().values
        ax.hist(vals1, bins=60, alpha=0.55, density=True, color=_PANEL1_COLOR, label=_ds1_label)
        ax.hist(vals2, bins=60, alpha=0.55, density=True, color=_PANEL2_COLOR, label=_ds2_label)
        ax.set_title(marker, fontsize=9)
        ax.set_xlabel("arcsinh", fontsize=7)
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Cross-Dataset: {_ds1_label} vs {_ds2_label} — Shared Markers", fontsize=12)
    fig.tight_layout()
    _cross_ds_dir = os.path.join(os.path.dirname(_sc1["outputs"]["matrix_corrected_path"]), "cross_dataset_visualization")
    os.makedirs(_cross_ds_dir, exist_ok=True)
    fig.savefig(os.path.join(_cross_ds_dir, f"cross_dataset_shared_markers_{_ds1_label}_vs_{_ds2_label}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── 2. Median expression heatmap per dataset ───────────────────────────────
    _med1 = df_ds1[_shared_markers].median().rename(_ds1_label)
    _med2 = df_ds2[_shared_markers].median().rename(_ds2_label)
    _med_df = pd.DataFrame([_med1, _med2])

    fig, ax = plt.subplots(figsize=(max(8, len(_shared_markers) * 0.5), 3))
    sns.heatmap(_med_df, ax=ax, cmap="viridis", annot=True, fmt=".2f",
                annot_kws={"size": 7}, linewidths=0.3)
    ax.set_title(f"Median Expression: {_ds1_label} vs {_ds2_label}", fontsize=11)
    ax.set_xlabel("Marker")
    fig.tight_layout()
    fig.savefig(os.path.join(_cross_ds_dir, f"cross_dataset_median_heatmap_{_ds1_label}_vs_{_ds2_label}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── 3. PCA variance by dataset factor ─────────────────────────────────────
    df_ds1["_dataset_label"] = _ds1_label
    df_ds2["_dataset_label"] = _ds2_label
    df_combined = pd.concat([df_ds1, df_ds2], ignore_index=True)

    _factors_avail = [c for c in ["plate", "group", "celltypes", "_dataset_label"] if c in df_combined.columns]
    _combined_panel = panel_filter or "AllPanels"
    var_combined = variation_pca(df_combined, markers=_shared_markers,
                                 group_col="_dataset_label",
                                 info_cols=_factors_avail,
                                 panel=_combined_panel)
    print(f"\nPCA Variance by factor (combined):")
    print(var_combined)

    del df_ds1, df_ds2, df_combined
    gc.collect()
    print(f"\n[DONE] Cross-dataset comparison saved to: {_cross_ds_dir}")